# MA-RUPA v2: Modality-Adaptive Multiscale Prototype Alignment Network
### Weakly-Supervised Video Anomaly Detection (WS-VAD) on Kaggle Dual GPU T4 x 2

**Mục tiêu Benchmark (Chuẩn Tạp chí Q1 / KLTN Xuất sắc — Lịch trình 10 Epochs):**
- **UCF-Crime:** Target Frame-level ROC-AUC **89.5% – 90.0%+**, AP **35.0%+**, Avg mAP@IoU **12.0% – 15.0%+**.
- **XD-Violence:** Target Frame-level AP **87.0% – 88.0%+**, ROC-AUC **95.0%+**, Avg mAP@IoU **18.0% – 20.0%+**.

**Các Đột phá Kiến trúc Cốt lõi (MA-RUPA v2):**
1. **SM-ASPM (`GradientSafeRUPA` v2):** Đa tạp Stiefel thích nghi mặt nạ mềm ($M: 16 \to 48$) dùng phép trực chuẩn hóa Polar Newton-Schulz (`torch.bmm`), Bộ nhớ Normal Toàn cục và hàm **Grassmannian Subspace Containment** ($1 - \|P_g t_0\|_2^2$) triệt tiêu hoàn toàn suy biến bậc 1 (`Orth = 1.00`).
2. **SA-EPTS & Stochastic Peak Erasure:** Hàm mượt bảo toàn biên kép Charbonnier + Mở rộng lân cận ngữ nghĩa CLIP (`SA-BEMIL`, $\cos(X_t, X_{t^*}) > 0.85$) + Xóa đỉnh ngẫu nhiên phá vỡ bão hòa MIL 1-frame.
3. **TASL-GSMA (`ModalityAdaptiveBranch` v2):** Nâng ngữ nghĩa âm thanh VGGish 128-D trực tiếp sang không gian CLIP Text 512-D + Chưng cất chéo có trọng số độ tin cậy + Khớp nối trực giao **Per-Snippet PCGrad**.
4. **CA-UCCF v2 & Unsupervised Logit Calibration:** Hợp nhất đa luồng theo tỉ số Tín hiệu trên Nền (Peak-to-Background SNR) và hiệu chuẩn phân vị monotonic không giám sát.

In [ ]:
# ========================= 1. SETUP ENVIRONMENT & GPU PREFLIGHT =========================
import os, sys, shutil, subprocess, time
from pathlib import Path
import torch

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO_DIR = WORK / "MA-RUPA"
ARTIFACTS = WORK / "rupa_artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Tùy chọn Branch GitHub:
# - "main": Mã nguồn MA-RUPA v2 mới nhất (SM-ASPM, SA-EPTS, TASL-GSMA, CA-UCCF)
# - "ucf-sota-87.11": Bản backup nguyên vẹn đạt 87.11% AUC trên UCF-Crime
REPO_URL = "https://github.com/Sharvuz/MA-RUPA.git"
REPO_REF = "main"

target_ref = REPO_REF if REPO_REF else "main"

if (REPO_DIR / ".git").exists():
    print(f"🔄 Đang đồng bộ commit mới nhất từ GitHub (Branch: {target_ref})...", flush=True)
    subprocess.run(["git", "fetch", "origin", target_ref], cwd=REPO_DIR, check=False)
    subprocess.run(["git", "checkout", target_ref], cwd=REPO_DIR, check=False)
    subprocess.run(["git", "reset", "--hard", f"origin/{target_ref}"], cwd=REPO_DIR, check=False)
elif not (REPO_DIR / "src").exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    print(f"📥 Đang clone repository từ GitHub (Branch: {target_ref})...", flush=True)
    clone_cmd = ["git", "clone", "--depth", "1", "--branch", target_ref, REPO_URL, str(REPO_DIR)]
    subprocess.run(clone_cmd, check=True)
else:
    print(f"✅ Đang sử dụng mã nguồn tại {REPO_DIR}", flush=True)

os.chdir(REPO_DIR)

# Cài đặt dependencies cần thiết
print("📦 Installing dependencies...", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ftfy", "regex", "einops==0.8.0", "ipdb", "packaging", "scikit-learn", "pandas", "scipy", "torchvision"
], check=True)

# Hardware & GPU Verification
print("\n" + "=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
gpu_count = torch.cuda.device_count()
print(f"Detected GPUs  : {gpu_count}")
for i in range(gpu_count):
    print(f"  GPU [{i}]: {torch.cuda.get_device_name(i)}")

if gpu_count >= 2:
    print("🔥 Dual GPU T4 x 2 detected! DataParallel with Interleaved Bags will be automatically activated.")
elif gpu_count == 1:
    print("⚡ Single GPU detected.")
else:
    print("⚠️ No GPU detected. Please set Accelerator = GPU T4 x 2 in Notebook Settings.")
print("=" * 60)


In [ ]:
# ========================= 2. RADAR: QUÉT DỮ LIỆU VISUAL & AUDIO + XỬ LÝ '#' -> '+' =========================
import re, zipfile
from collections import defaultdict
import numpy as np
import pandas as pd

print("🔍 Đang quét toàn bộ /kaggle/input và /kaggle/working để lập bản đồ feature (.npy)...")

# 1. Tự động giải nén file .zip nếu có
zip_files = list(INPUT.rglob("*.zip"))
if zip_files:
    extract_dir = WORK / "extracted_features"
    extract_dir.mkdir(parents=True, exist_ok=True)
    print(f"📦 Tìm thấy {len(zip_files)} file zip. Đang kiểm tra giải nén...")
    for zf in zip_files:
        dest = extract_dir / zf.stem
        if not dest.exists() or not any(dest.iterdir()):
            dest.mkdir(parents=True, exist_ok=True)
            print(f"  Unzipping {zf.name} -> {dest.name}...")
            with zipfile.ZipFile(zf, 'r') as archive:
                archive.extractall(dest)

# 2. Quét tất cả các file .npy trên hệ thống
search_roots = [INPUT, WORK]
all_npy_files = []
for root in search_roots:
    if root.exists():
        all_npy_files.extend(root.rglob("*.npy"))

print(f"📡 Radar phát hiện tổng cộng {len(all_npy_files)} tệp tin .npy trên hệ thống!")

# 3. Lập bản đồ tra cứu đa chiều (Hỗ trợ đổi tên '#' -> '+' trên Kaggle)
file_map = {}
audio_map = {}
for p in all_npy_files:
    fname = p.name
    full_path = str(p.resolve())
    variants = {
        fname,
        fname.replace('+', '#'),
        fname.replace('#', '+'),
        fname.replace('+', '_').replace('#', '_'),
    }
    for v in variants:
        file_map[v] = full_path
    if '__vggish' in fname or '_vggish' in fname or 'audio' in str(p).lower():
        base_aud = fname.replace('__vggish.npy', '').replace('_vggish.npy', '').replace('.npy', '')
        for v in {
            base_aud,
            base_aud.replace('+', '#'),
            base_aud.replace('#', '+'),
            base_aud.replace('+', '_').replace('#', '_'),
        }:
            audio_map[v] = full_path

def sync_csv_list(dataset_name, csv_rel_path, output_name):
    """Đồng bộ đường dẫn CSV gốc sang đường dẫn thực tế trên Kaggle."""
    input_csv = REPO_DIR / csv_rel_path
    output_csv = ARTIFACTS / output_name
    if not input_csv.exists():
        raise FileNotFoundError(f"Không tìm thấy file danh sách mẫu: {input_csv}")

    df = pd.read_csv(input_csv)
    resolved_paths = []
    missing_files = []
    for orig_path in df['path'].astype(str):
        base_name = Path(orig_path).name
        candidates = [
            base_name,
            base_name.replace('#', '+'),
            base_name.replace('+', '#'),
            base_name.replace('#', '_'),
            base_name.replace('#', ''),
        ]
        match = next((file_map[c] for c in candidates if c in file_map), None)
        if match:
            resolved_paths.append(match)
        else:
            missing_files.append(base_name)
            resolved_paths.append(orig_path)

    if missing_files:
        print(f"⚠️ Cảnh báo: Thiếu {len(missing_files)}/{len(df)} files cho {output_name}. Ví dụ: {missing_files[:3]}")
    else:
        print(f"✅ Đồng bộ 100% {len(df)} file cho {output_name} thành công!")

    df['path'] = resolved_paths
    df.to_csv(output_csv, index=False)
    return output_csv

def build_xd_audio_csv(visual_csv_path, output_name):
    """Tạo bảng ánh xạ (path, audio_path) cho XD-Violence từ các tệp VGGish 128-D."""
    output_csv = ARTIFACTS / output_name
    df = pd.read_csv(visual_csv_path)
    audio_paths = []
    matched = 0
    for v_path in df['path'].astype(str):
        v_stem = Path(v_path).stem
        base_id = re.sub(r'__\d+$', '', v_stem)
        aud_match = audio_map.get(base_id)
        if not aud_match:
            for cand in [base_id.replace('#', '+'), base_id.replace('+', '#'), base_id.replace('+', '_').replace('#', '_')]:
                if cand in audio_map:
                    aud_match = audio_map[cand]
                    break
        if aud_match:
            matched += 1
            audio_paths.append(aud_match)
        else:
            audio_paths.append("")
    out_df = pd.DataFrame({'path': df['path'], 'audio_path': audio_paths})
    out_df.to_csv(output_csv, index=False)
    print(f"🎵 Ánh xạ Audio {output_name}: ghép cặp thành công {matched}/{len(df)} ({matched / max(1, len(df)) * 100:.1f}%) clips!")
    return output_csv

# Đồng bộ danh sách cho cả 2 tập dữ liệu
ucf_train_csv = sync_csv_list("ucf", "list/ucf_CLIP_rgb.csv", "ucf_train.csv")
ucf_test_csv = sync_csv_list("ucf", "list/ucf_CLIP_rgbtest.csv", "ucf_test.csv")
xd_train_csv = sync_csv_list("xd", "list/xd_CLIP_rgb.csv", "xd_train.csv")
xd_test_csv = sync_csv_list("xd", "list/xd_CLIP_rgbtest.csv", "xd_test.csv")

# Tạo danh sách ánh xạ Audio cho XD-Violence (khắc phục triệt để NameError xd_audio_train_csv)
xd_audio_train_csv = build_xd_audio_csv(xd_train_csv, "xd_audio_train.csv")
xd_audio_test_csv = build_xd_audio_csv(xd_test_csv, "xd_audio_test.csv")

# Kiểm tra thử nghiệm Visual & Audio Feature Preflight
sample_p = Path(pd.read_csv(ucf_train_csv).iloc[0]['path'])
if sample_p.exists():
    sample_feat = np.load(sample_p, mmap_mode='r')
    print(f"\n🎯 Visual Feature Preflight: {sample_p.name} | Shape: {sample_feat.shape}")

if audio_map:
    sample_aud = Path(next(iter(audio_map.values())))
    if sample_aud.exists():
        sample_aud_feat = np.load(sample_aud, mmap_mode='r')
        print(f"🎵 Audio Feature Preflight : {sample_aud.name} | Shape: {sample_aud_feat.shape} | Tổng số keys: {len(audio_map)}")


## 3. UCF-Crime Training (MA-RUPA v2 | 10 Epochs on GPU T4 x 2)

- **Mục tiêu hiệu năng:** **89.5% – 90.0% Frame-level ROC-AUC** (`AP >= 35.0%`, `Avg mAP@IoU >= 12.0% - 15.0%`).
- **Cơ chế MA-RUPA v2:** Tích hợp **SM-ASPM** ($M: 16 \to 48$ với Grassmannian Subspace Containment), **SA-EPTS** (Charbonnier bảo toàn biên + CLIP-Affinity Halo Expansion + Stochastic Peak Erasure), **TASL** trên Motion Residuals ($M_1, M_2$) và **CA-UCCF v2**.

In [ ]:
# ========================= 3. TRAIN UCF-CRIME (MA-RUPA v2 | GPU T4 x 2) =========================
import subprocess, sys

# Tự động đồng bộ commit mới nhất từ GitHub
subprocess.run(["git", "fetch", "origin", "main"], cwd=REPO_DIR, check=False)
subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=REPO_DIR, check=False)

UCF_BATCH_SIZE = 64
UCF_EPOCHS = 10
UCF_LR = 7e-5
SEED = 234

ucf_model_path = ARTIFACTS / "model_ucf_best.pth"
ucf_checkpoint_path = ARTIFACTS / "checkpoint_ucf.pth"

cmd_train_ucf = [
    sys.executable, "-u", "src/ucf_train.py",
    "--train-list", str(ucf_train_csv),
    "--test-list", str(ucf_test_csv),
    "--gt-path", "list/gt_ucf.npy",
    "--gt-segment-path", "list/gt_segment_ucf.npy",
    "--gt-label-path", "list/gt_label_ucf.npy",
    "--batch-size", str(UCF_BATCH_SIZE),
    "--lr", str(UCF_LR),
    "--max-epoch", str(UCF_EPOCHS),
    "--curriculum-epochs", "3.0",
    "--max-omega", "0.20",
    "--sampling", "topk",
    "--normal_selection_ratio", "0.80",
    "--t_w", "0.10",
    "--topk-ratio", "16",              # ~6.25% snippets (tối ưu cho sự kiện CCTV ngắn)
    "--topk-ratio-clasm", "16",
    "--sample-one-crop-per-video", "False",
    "--use-aspm", "True",              # SM-ASPM: Đa tạp Stiefel thích nghi M: 16 -> 48
    "--aspm-max-rank", "48",
    "--use-epts", "True",              # SA-EPTS: Charbonnier + Halo Expansion + Peak Erasure
    "--use-tasl", "True",              # TASL: Nâng ngữ nghĩa chuyển động neo theo CLIP Text
    "--use-pcgrad", "True",            # PCGrad: Khử xung đột gradient đa nhánh
    "--snippet-dropout", "0.15",
    "--lambda-aux", "0.15",
    "--lambda-tasl", "0.10",
    "--lambda-smooth", "0.06",
    "--loss2_weight", "1.2",
    "--lambda-consistency", "0.15",
    "--lambda-bkg", "0.5",
    "--lambda-gather", "0.12",
    "--lambda-orth", "0.05",
    "--lambda-residual", "0.05",
    "--lambda-gate", "0.01",
    "--seed", str(SEED),
    "--model-path", str(ucf_model_path),
    "--checkpoint-path", str(ucf_checkpoint_path),
    "--temp", "5.0",
    "--smooth-eval", "True",
    "--smooth-sigma", "0.50"
]
print("🚀 Bắt đầu huấn luyện End-to-End MA-RUPA v2 trên UCF-Crime (10 Epochs) với GPU T4x2...\n", flush=True)

process = subprocess.Popen(
    cmd_train_ucf, cwd=REPO_DIR, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1
)

for line in process.stdout:
    print(line, end="")
    sys.stdout.flush()

ret_code = process.wait()
if ret_code != 0:
    raise RuntimeError(f"❌ Training UCF-Crime thất bại với mã lỗi {ret_code}")
else:
    print(f"\n✅ Huấn luyện UCF-Crime thành công! Model tốt nhất lưu tại: {ucf_model_path}")


In [ ]:
# ========================= 4. EVALUATE UCF-CRIME =========================
cmd_eval_ucf = [
    sys.executable, "-u", "src/ucf_test.py",
    "--test-list", str(ucf_test_csv),
    "--gt-path", "list/gt_ucf.npy",
    "--gt-segment-path", "list/gt_segment_ucf.npy",
    "--gt-label-path", "list/gt_label_ucf.npy",
    "--model-path", str(ucf_model_path),
    "--temp", "5.0",
    "--smooth-eval", "True",
    "--smooth-sigma", "0.50"
]

print("🔍 Đang đánh giá model UCF-Crime tốt nhất với CA-UCCF v2 & DODO v2...", flush=True)
subprocess.run(cmd_eval_ucf, cwd=REPO_DIR, check=True)


## 4. XD-Violence Training (MA-RUPA v2 Audio-Visual | 10 Epochs on GPU T4 x 2)

- **Mục tiêu hiệu năng:** **87.0% – 88.0%+ Frame-level Average Precision (AP)** (`ROC-AUC >= 95.0%`, `Avg mAP@IoU >= 18.0% - 20.0%`).
- **Cơ chế MA-RUPA v2:** Kết hợp Visual CLIP (512-D) và Audio VGGish (128-D) được nâng ngữ nghĩa qua **TASL** (`Audio-to-CLIP-Text CLASM` + chưng cất chéo có trọng số độ tin cậy), bảo vệ backbone bằng **PCGrad**, mở rộng không gian con bình thường qua **SM-ASPM** ($M: 16 \to 48, \omega_{\max} = 0.40$) và hợp nhất đã hiệu chuẩn phân vị qua **CA-UCCF v2**.

In [ ]:
# ========================= 5. TRAIN XD-VIOLENCE (MA-RUPA v2 | GPU T4 x 2) =========================
import subprocess, sys

# Tự động đồng bộ commit mới nhất từ GitHub
subprocess.run(["git", "fetch", "origin", "main"], cwd=REPO_DIR, check=False)
subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=REPO_DIR, check=False)

XD_BATCH_SIZE = 64
XD_EPOCHS = 10
XD_LR = 1.5e-5
SEED = 234

xd_model_path = ARTIFACTS / "model_xd_best.pth"
xd_checkpoint_path = ARTIFACTS / "checkpoint_xd.pth"

cmd_train_xd = [
    sys.executable, "-u", "src/xd_train.py",
    "--train-list", str(xd_train_csv),
    "--test-list", str(xd_test_csv),
    "--audio-train-list", str(xd_audio_train_csv),
    "--audio-test-list", str(xd_audio_test_csv),
    "--gt-path", "list/gt.npy",
    "--gt-segment-path", "list/gt_segment.npy",
    "--gt-label-path", "list/gt_label.npy",
    "--batch-size", str(XD_BATCH_SIZE),
    "--lr", str(XD_LR),
    "--max-epoch", str(XD_EPOCHS),
    "--curriculum-epochs", "3.0",
    "--max-omega", "0.20",
    "--sampling", "topk",
    "--normal_selection_ratio", "0.80",
    "--topk-ratio", "8",
    "--topk-ratio-clasm", "8",
    "--sample-one-crop-per-video", "False",
    "--t_w", "0.45",
    "--routing-det-weight", "0.45",
    "--routing-sem-weight", "0.42",
    "--routing-aux-weight", "0.08",
    "--routing-rec-weight", "0.05",
    "--use-aspm", "True",              # SM-ASPM: Đa tạp Stiefel thích nghi M: 16 -> 48
    "--aspm-max-rank", "48",
    "--use-epts", "True",              # SA-EPTS: Charbonnier + Halo Expansion + Block K-Dropout
    "--use-tasl", "True",              # TASL + Sparse Semantic Subspace Audio-Visual Alignment
    "--use-pcgrad", "True",            # PCGrad: Khử xung đột gradient Audio-Visual & RUPA
    "--snippet-dropout", "0.15",
    "--lambda-aux", "0.18",
    "--lambda-tasl", "0.25",
    "--lambda-smooth", "0.06",
    "--loss2_weight", "1.4",
    "--lambda-consistency", "0.15",
    "--lambda-bkg", "0.5",
    "--lambda-gather", "0.12",
    "--lambda-orth", "0.05",
    "--lambda-residual", "0.05",
    "--lambda-gate", "0.01",
    "--seed", str(SEED),
    "--model-path", str(xd_model_path),
    "--checkpoint-path", str(xd_checkpoint_path),
    "--temp", "1.0",
    "--smooth-eval", "True",
    "--smooth-sigma", "0.50"
]
print("🚀 Bắt đầu huấn luyện End-to-End MA-RUPA v2 Multimodal (Dual-Space + Sparse Align) trên XD-Violence (10 Epochs)...\n", flush=True)

process = subprocess.Popen(
    cmd_train_xd, cwd=REPO_DIR, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1
)

for line in process.stdout:
    print(line, end="")
    sys.stdout.flush()

ret_code = process.wait()
if ret_code != 0:
    raise RuntimeError(f"❌ Training XD-Violence thất bại với mã lỗi {ret_code}")
else:
    print(f"\n✅ Huấn luyện XD-Violence thành công! Model tốt nhất lưu tại: {xd_model_path}")


In [ ]:
# ========================= 6. EVALUATE XD-VIOLENCE =========================
cmd_eval_xd = [
    sys.executable, "-u", "src/xd_test.py",
    "--test-list", str(xd_test_csv),
    "--audio-test-list", str(xd_audio_test_csv),
    "--gt-path", "list/gt.npy",
    "--gt-segment-path", "list/gt_segment.npy",
    "--gt-label-path", "list/gt_label.npy",
    "--model-path", str(xd_model_path),
    "--temp", "1.0",
    "--smooth-eval", "True",
    "--smooth-sigma", "0.50"
]

print("🔍 Đang đánh giá model XD-Violence tốt nhất với CA-UCCF v2 & DODO v2...", flush=True)
subprocess.run(cmd_eval_xd, cwd=REPO_DIR, check=True)


## 5. Ablation Studies (Phục vụ bài báo Q1 / KLTN)

Các cờ dòng lệnh cho phép bạn thử nghiệm và tái tạo đầy đủ bảng Ablation cho cả **MA-RUPA v1 & v2**:

| Cấu hình nghiên cứu | Cờ dòng lệnh | Ý nghĩa nghiên cứu |
|---|---|---|
| **w/o SM-ASPM (Fixed $M=16$)** | `--use-aspm False` | Đánh giá đóng góp của Đa tạp Stiefel thích nghi $M: 16 \to 48$ |
| **w/o SA-EPTS (Standard TV)** | `--use-epts False` | Đánh giá đóng góp của Charbonnier + CLIP Halo Expansion + Peak Erasure |
| **w/o TASL (Raw VGGish Proj)** | `--use-tasl False` | Đánh giá đóng góp của Nâng ngữ nghĩa Audio neo theo CLIP Text |
| **w/o PCGrad** | `--use-pcgrad False` | Đánh giá đóng góp của Khớp nối trực giao Gradient đa nhánh |
| **w/o MDTP** | `--no-mdtp` | Đánh giá vai trò của Multi-Scale Dilated Temporal Pyramid |
| **w/o Audio / Motion** | `--no-audio` | Đánh giá vai trò của nhánh phụ trợ đa phương thức |
| **w/o RUPA** | `--no-rupa` | Đánh giá vai trò của tái tạo không gian con và DNP |
| **w/o Safe Gate** | `--no-safegate` | Đánh giá vai trò của cổng định tuyến QuadSafeGate |

In [ ]:
# Ví dụ chạy thí nghiệm Ablation Study cho bài báo / KLTN:
# !python src/xd_train.py --train-list /kaggle/working/rupa_artifacts/xd_train.csv --test-list /kaggle/working/rupa_artifacts/xd_test.csv --use-aspm False
# !python src/ucf_train.py --train-list /kaggle/working/rupa_artifacts/ucf_train.csv --test-list /kaggle/working/rupa_artifacts/ucf_test.csv --use-epts False
